# 🩺 SkinDisNet — Skin Disease **Object Detection** (Grad-CAM Weakly-Supervised)
### Computer Vision Assignment | Google Colab

| | |
|---|---|
| **Task** | Object Detection — bounding box + disease class on skin images |
| **Approach** | EfficientNetB0 classifier → **Grad-CAM** localization → bounding box |
| **Why this approach** | The dataset has only image-level labels (no boxes), so we *derive* boxes from where the model looks |
| **Platform** | Google Colab (T4 GPU) |

---
### 📦 What this notebook produces (your submission)
A single **`SkinDisNet_Detection_Submission.zip`** containing:
- `detected_images/` — skin images with a **bounding box + predicted class + confidence** drawn on them
- `explanation.txt` — short write-up of the approach and tools

### ⚡ How to run
1. Put your dataset `.zip` in your **Google Drive (My Drive root)**.
2. Enable GPU: **Runtime → Change runtime type → T4 GPU**.
3. **If you have already run any cell this session: Runtime → Restart session first.**
   (Step 1 switches to the stable Keras 2 API, which only works on a fresh start.)
4. **Run all cells top to bottom** (Step 2 will ask you to authorize Drive).
5. The last cell builds and **downloads** the submission zip automatically.

## ⚙️ Step 1: GPU Check & Imports

In [ ]:
# ── Use stable Keras 2 (avoids the Keras 3 "dtype='string'" type-promotion bug) ──
# Colab now ships Keras 3, whose model.fit breaks for this model. Setting
# TF_USE_LEGACY_KERAS routes tf.keras back to the stable Keras 2 API this
# notebook is written for. This MUST run before TensorFlow is imported.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
!pip install -q tf-keras

import tensorflow as tf
print(f"TensorFlow : {tf.__version__}")
try:
    import tf_keras
    print(f"Keras      : {tf_keras.__version__}   (stable Keras 2 ✅)")
except Exception:
    print("Keras      : legacy Keras 2 active ✅")
gpus = tf.config.list_physical_devices('GPU')
print(f"✅ GPU: {gpus[0].name}" if gpus else "⚠️  No GPU — Runtime > Change runtime type > T4 GPU")

import numpy as np, shutil, glob, random, json
import matplotlib.pyplot as plt
import cv2
from PIL import Image

from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

random.seed(42); np.random.seed(42)
print("✅ Libraries loaded.")

## 📂 Step 2: Mount Google Drive & Unzip Your Dataset

Set `ZIP_NAME` below to the name of the zip you uploaded to **My Drive**. The cell mounts
Drive, unzips it, and auto-detects the folder that holds the class sub-folders.

Your zip should contain **one folder per class**, each holding that class's images, e.g.:

```
YourDataset.zip
└── (any folders) / Atopic Dermatitis / *.jpg
                  / Eczema            / *.jpg
                  / Scabies           / *.jpg
                  ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Set this to your zip's name in My Drive ──────────────────────────────
ZIP_NAME = "SkinDisNet.zip"        # ← change to your file name
ZIP_PATH = f"/content/drive/MyDrive/{ZIP_NAME}"
# If your zip is inside a subfolder of Drive, use the full path, e.g.:
# ZIP_PATH = "/content/drive/MyDrive/MyFolder/SkinDisNet.zip"
# ──────────────────────────────────────────────────────────────────────────

assert os.path.exists(ZIP_PATH), f"❌ Not found: {ZIP_PATH}\n   Check ZIP_NAME / the path in My Drive."

EXTRACT_PATH = "/content/dataset"
if not os.path.exists(EXTRACT_PATH):
    print(f"📦 Extracting {ZIP_NAME}...")
    shutil.unpack_archive(ZIP_PATH, EXTRACT_PATH)
    print("✅ Extracted.")
else:
    print("✅ Already extracted — skipping.")

# Auto-detect the folder whose sub-folders directly contain images (= class folders)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
DATASET_PATH = None
for root, dirs, _ in os.walk(EXTRACT_PATH):
    img_subdirs = [d for d in dirs
                   if any(f.lower().endswith(IMG_EXT)
                          for f in os.listdir(os.path.join(root, d)))]
    if len(img_subdirs) >= 2:
        DATASET_PATH = root
        break
assert DATASET_PATH, "❌ Could not find class folders inside the zip. Expected one folder of images per class."

CLASS_NAMES = sorted(d for d in os.listdir(DATASET_PATH)
                     if os.path.isdir(os.path.join(DATASET_PATH, d))
                     and any(f.lower().endswith(IMG_EXT)
                             for f in os.listdir(os.path.join(DATASET_PATH, d))))
NUM_CLASSES = len(CLASS_NAMES)
print(f"\n📁 Dataset folder: {DATASET_PATH}")
print(f"✅ Classes ({NUM_CLASSES}):")
for c in CLASS_NAMES:
    n = len([f for f in os.listdir(os.path.join(DATASET_PATH, c)) if f.lower().endswith(IMG_EXT)])
    print(f"   {c:<30} → {n:>4} images")

## 🧠 Step 3: Train EfficientNetB0 Classifier (needed for Grad-CAM)

Grad-CAM localizes the lesion using a *skin-trained* model, so we first train a classifier. The base is built with `input_tensor` so it is a **single flat graph** — this makes Grad-CAM reliable. Only the head is trained (fast); reduce `EPOCHS` if you are short on time.

In [ ]:

IMG_SIZE, BATCH_SIZE, SEED = 224, 32, 42

# Keras 3 native data pipeline (the legacy ImageDataGenerator triggers a
# "dtype='string'" type-promotion bug under Keras 3, so we use tf.data instead).
from tensorflow.keras.utils import image_dataset_from_directory

train_ds = image_dataset_from_directory(
    DATASET_PATH, validation_split=0.2, subset='training', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical')
val_ds = image_dataset_from_directory(
    DATASET_PATH, validation_split=0.2, subset='validation', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical')

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

# Augmentation in the data pipeline (keeps the model graph flat for Grad-CAM).
# EfficientNet expects raw 0-255 input (its normalization is built in), so no rescaling.
AUTOTUNE = tf.data.AUTOTUNE
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
])
train_ds = train_ds.map(lambda x, y: (augment(x, training=True), y),
                        num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# Flat model so Grad-CAM can reach the last conv layer by name
inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
base = EfficientNetB0(include_top=False, input_tensor=inputs, weights='imagenet')
base.trainable = False
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs, outputs)
model.compile(keras.optimizers.Adam(1e-3), 'categorical_crossentropy', ['accuracy'])

EPOCHS = 15   # ← lower to ~8 if short on time; Grad-CAM only needs a decent model
cbs = [EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
       ModelCheckpoint('/content/clf_best.keras', monitor='val_accuracy', save_best_only=True)]
print("🚀 Training classifier...")
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cbs, verbose=1)
print(f"\n✅ Done — best val acc: {max(model.history.history['val_accuracy'])*100:.2f}%")

## 🔥 Step 4: Grad-CAM → Bounding Box Detector

`make_gradcam_heatmap` gets the heatmap over the last conv layer (`top_activation`). `heatmap_to_bbox` thresholds it and takes the largest connected region as the bounding box. `detect` ties it together: predict class → localize → draw box + label + confidence.

In [ ]:
LAST_CONV = 'top_activation'   # EfficientNetB0 final conv activation (7x7x1280)

def make_gradcam_heatmap(img_array, model, last_conv_name, pred_index=None):
    grad_model = keras.models.Model(model.inputs,
        [model.get_layer(last_conv_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = tf.squeeze(conv_out @ pooled[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_index, preds.numpy()[0]

def heatmap_to_bbox(heatmap, w, h, thresh=0.5):
    hm = cv2.resize(heatmap, (w, h))
    binary = cv2.threshold(np.uint8(255 * hm), int(thresh * 255), 255, cv2.THRESH_BINARY)[1]
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    return cv2.boundingRect(max(contours, key=cv2.contourArea))   # x, y, w, h

def detect(image_path, save_path=None, show=True):
    img  = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    rgb  = np.array(img)
    batch = np.expand_dims(preprocess_input(rgb.astype(np.float32)), 0)

    heatmap, idx, probs = make_gradcam_heatmap(batch, model, LAST_CONV)
    pred_class, conf = CLASS_NAMES[idx], probs[idx] * 100
    box = heatmap_to_bbox(heatmap, IMG_SIZE, IMG_SIZE)

    overlay = cv2.applyColorMap(np.uint8(255 * cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))),
                                cv2.COLORMAP_JET)[:, :, ::-1]
    blended = np.uint8(0.6 * rgb + 0.4 * overlay)

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(rgb); ax[0].set_title('Detection', fontsize=11, fontweight='bold'); ax[0].axis('off')
    if box:
        x, y, bw, bh = box
        import matplotlib.patches as patches
        ax[0].add_patch(patches.Rectangle((x, y), bw, bh, lw=2.5, edgecolor='lime', facecolor='none'))
        ax[0].text(x, max(y - 6, 8), f"{pred_class} {conf:.0f}%", color='black', fontsize=10,
                   bbox=dict(facecolor='lime', edgecolor='none', pad=1.5))
    ax[1].imshow(blended); ax[1].set_title('Grad-CAM heatmap', fontsize=11, fontweight='bold'); ax[1].axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches='tight')
    (plt.show() if show else plt.close(fig))
    return pred_class, conf, box

print("✅ Detector ready.")

## 🖼️ Step 5: Run Detection on Sample Images → Save Results

Detects on a few images per class and writes annotated images into the submission folder. Increase `N_PER_CLASS` for more results.

In [ ]:
SUB_DIR = '/content/submission'
IMG_OUT = os.path.join(SUB_DIR, 'detected_images')
shutil.rmtree(SUB_DIR, ignore_errors=True)
os.makedirs(IMG_OUT, exist_ok=True)

N_PER_CLASS = 3
records = []
for cls in CLASS_NAMES:
    files = sorted(glob.glob(os.path.join(DATASET_PATH, cls, '*')))
    files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
    for i, f in enumerate(random.sample(files, min(N_PER_CLASS, len(files)))):
        out = os.path.join(IMG_OUT, f"detected_{cls.replace(' ', '_')}_{i+1}.png")
        pred, conf, box = detect(f, save_path=out, show=True)
        records.append((cls, os.path.basename(f), pred, conf, box))
        print(f"  {cls:<28} → predicted {pred:<28} {conf:5.1f}%  box={box}")

print(f"\n✅ Saved {len(records)} detected images to {IMG_OUT}")

## 📝 Step 6: Write Explanation & Build + Download the Submission Zip

In [ ]:
explanation = '''SKIN DISEASE OBJECT DETECTION - APPROACH & TOOLS
=====================================================

TASK
----
Detect (localize + classify) skin-disease lesions in clinical images, drawing a
bounding box and predicted disease class on each image.

APPROACH (weakly-supervised detection via Grad-CAM)
---------------------------------------------------
The dataset provides only image-level class labels (one disease label per image) and
NO bounding-box annotations. Standard supervised object detectors (YOLO, Faster R-CNN)
cannot be trained without boxes, so a weakly-supervised localization approach was used:

1. Train an EfficientNetB0 image classifier (transfer learning from ImageNet) on the
   skin-disease classes.
2. For each test image, run Grad-CAM (Gradient-weighted Class Activation Mapping) on
   the last convolutional layer to produce a heatmap of the regions the model used to
   make its prediction - i.e. the lesion area.
3. Convert the heatmap into a bounding box by thresholding it and taking the largest
   connected region (OpenCV contour detection).
4. Draw the bounding box, predicted disease class, and confidence on the image.

This yields object-detection-style outputs (box + label + confidence) directly from a
classifier, without any manual annotation.

TOOLS USED
----------
- Python 3, Google Colab (T4 GPU)
- TensorFlow / Keras  - EfficientNetB0 classifier (transfer learning)
- Grad-CAM            - lesion localization from classifier gradients
- OpenCV (cv2)        - heatmap thresholding and bounding-box extraction
- NumPy, Matplotlib   - array handling and rendering the annotated images

OUTPUTS
-------
- detected_images/ : skin images annotated with bounding box + predicted class + confidence
- explanation.txt  : this file
'''

with open(os.path.join(SUB_DIR, 'explanation.txt'), 'w') as f:
    f.write(explanation)

zip_base = '/content/SkinDisNet_Detection_Submission'
shutil.make_archive(zip_base, 'zip', SUB_DIR)
zip_path = zip_base + '.zip'
print(f"✅ Submission zip: {zip_path}")
print(f"   Contains: {len(os.listdir(IMG_OUT))} detected images + explanation.txt")

# Also back up the zip to your Google Drive
drive_dir = '/content/drive/MyDrive/SkinDisNet_Detection/'
os.makedirs(drive_dir, exist_ok=True)
shutil.copy(zip_path, drive_dir)
print(f"✅ Backed up to Drive: {drive_dir}")

# Download the single zip to your computer
from google.colab import files as colab_files
colab_files.download(zip_path)

## ✅ Summary

| Item | Details |
|---|---|
| **Task** | Object detection (bounding box + disease class) on skin images |
| **Approach** | EfficientNetB0 classifier → Grad-CAM heatmap → bounding box |
| **Why** | Dataset has no bounding-box annotations; boxes are derived from the model's attention |
| **Tools** | TensorFlow/Keras, Grad-CAM, OpenCV, NumPy, Matplotlib, Colab GPU |
| **Deliverable** | `SkinDisNet_Detection_Submission.zip` (detected images + explanation.txt) |

**To submit:** the zip auto-downloads in Step 6 (also saved to `MyDrive/SkinDisNet_Detection/`). Upload that single zip to the assignment.